# Lab 7b: Autoencoder dan Embedding

**Pendamping Bab 07 (Alat Pendukung Ringan).**

Tujuan: melatih *convolutional autoencoder* unsupervised pada CIFAR-10 (tanpa memakai label), memvisualisasikan ruang bottleneck dengan t-SNE atau UMAP, dan menguji varian *denoising autoencoder*. Autoencoder berfungsi sebagai *alat pendukung riset*: ia adalah pintu ke representation learning dan jembatan ke keluarga model generatif (VAE, GAN, Diffusion) yang dibahas peta-nya di Bab 09.

**Prasyarat:** Lab 1 (baseline CNN), Section 2.6 (representasi fitur).

**Estimasi:** 3-4 jam.

## 0. Setup

### Lokal
```bash
pip install -e .
```

### Google Colab
Jalankan sel kode **0a. Setup Colab** di bawah ini. Setelah selesai, lanjut ke **1. Setup**.

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/muhammad-zainal-muttaqin/ModulePembelajaran.git'
    REPO_DIR = '/content/ModulePembelajaran'

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    else:
        print('Repo sudah ada, skip clone.')

    %cd {REPO_DIR}/ModulePembelajaran/template_repo
    !pip install -e .
    print('\nSetup Colab selesai. Working dir:', os.getcwd())
else:
    print('Environment lokal. Lewati setup Colab.')

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from src.models import SimpleAutoencoder

torch.manual_seed(42); np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 2. Data: CIFAR-10 tanpa label

Label akan kita simpan hanya untuk tujuan visualisasi bottleneck di akhir - training sepenuhnya unsupervised (hanya input, target adalah input itu sendiri).

In [ ]:
tfm = transforms.ToTensor()
train_ds = datasets.CIFAR10(root='./data', train=True,  download=True, transform=tfm)
val_ds   = datasets.CIFAR10(root='./data', train=False, download=True, transform=tfm)
dl_tr = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2)
dl_va = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=2)
print('train:', len(train_ds), 'val:', len(val_ds))

## 3. Training autoencoder standar

In [ ]:
def train_ae(model, dl_tr, dl_va, epochs=10, lr=1e-3, denoising_std=0.0):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    hist = {'tr': [], 'va': []}
    for epoch in range(epochs):
        model.train(); running = 0.0
        for Xb, _ in dl_tr:
            Xb = Xb.to(device)
            inp = Xb + denoising_std * torch.randn_like(Xb) if denoising_std > 0 else Xb
            out = model(inp)
            loss = F.mse_loss(out, Xb)
            opt.zero_grad(); loss.backward(); opt.step()
            running += loss.item() * len(Xb)
        model.eval(); val_running = 0.0
        with torch.no_grad():
            for Xb, _ in dl_va:
                Xb = Xb.to(device)
                val_running += F.mse_loss(model(Xb), Xb).item() * len(Xb)
        hist['tr'].append(running / len(dl_tr.dataset))
        hist['va'].append(val_running / len(dl_va.dataset))
        print(f'ep{epoch+1:2d}  tr={hist["tr"][-1]:.4f}  va={hist["va"][-1]:.4f}')
    return hist

ae = SimpleAutoencoder(image_channels=3, bottleneck_dim=32).to(device)
hist_clean = train_ae(ae, dl_tr, dl_va, epochs=10)
plt.plot(hist_clean['tr'], label='train'); plt.plot(hist_clean['va'], label='val')
plt.legend(); plt.xlabel('epoch'); plt.ylabel('MSE'); plt.title('AE standar')
plt.show()

## 4. Rekonstruksi kualitatif

In [ ]:
ae.eval()
Xb, _ = next(iter(dl_va))
with torch.no_grad():
    recon = ae(Xb.to(device)).cpu()
fig, ax = plt.subplots(2, 8, figsize=(14, 3.5))
for i in range(8):
    ax[0, i].imshow(Xb[i].permute(1, 2, 0)); ax[0, i].axis('off')
    ax[1, i].imshow(recon[i].permute(1, 2, 0)); ax[1, i].axis('off')
ax[0, 0].set_title('input', loc='left'); ax[1, 0].set_title('recon', loc='left')
plt.tight_layout(); plt.show()

## 5. Visualisasi bottleneck dengan t-SNE

Apakah bottleneck 32-D yang dipelajari unsupervised menunjukkan cluster yang mengikuti kelas? Kita tidak mengharapkan separasi sempurna (AE standar tidak dilatih dengan label), tapi pola cluster yang lemah sekalipun menunjukkan AE menangkap struktur semantik.

In [ ]:
from sklearn.manifold import TSNE

embeds, labels = [], []
ae.eval()
with torch.no_grad():
    for Xb, yb in dl_va:
        z = ae.encode(Xb.to(device)).cpu().numpy()
        embeds.append(z); labels.append(yb.numpy())
        if sum(len(l) for l in labels) >= 2000:
            break
Z = np.concatenate(embeds)[:2000]
y = np.concatenate(labels)[:2000]

Z2 = TSNE(n_components=2, perplexity=30, init='pca', random_state=42).fit_transform(Z)
plt.figure(figsize=(7, 6))
scatter = plt.scatter(Z2[:, 0], Z2[:, 1], c=y, cmap='tab10', s=6)
plt.legend(*scatter.legend_elements(), title='class', loc='best', fontsize=7)
plt.title('t-SNE bottleneck AE (2000 sampel)')
plt.show()

## 6. Varian: Denoising Autoencoder

Input dirusak noise gaussian; target tetap gambar bersih. DAE sering memberi embedding yang lebih robust - cek apakah t-SNE-nya menunjukkan cluster lebih tegas.

In [ ]:
dae = SimpleAutoencoder(image_channels=3, bottleneck_dim=32).to(device)
hist_dae = train_ae(dae, dl_tr, dl_va, epochs=10, denoising_std=0.2)

## 7. Peta ke keluarga generatif

Autoencoder biasa memetakan `x → z → x̂` - bottleneck `z` adalah representasi, tidak lebih. **Variational Autoencoder** (VAE, Kingma & Welling 2013) menambahkan distribusi ke bottleneck: encoder mengeluarkan `(μ, σ)` untuk distribusi Gaussian, decoder mengambil sampel dari distribusi itu. Loss VAE (ELBO) = *reconstruction loss* + *KL divergence* terhadap prior Gaussian standar. Akibatnya: `z` tidak hanya cocok buat rekonstruksi, tapi juga membentuk ruang latent yang bisa disampel untuk **generatif** - menyalakan model untuk menciptakan gambar baru dengan men-sampling `z` dari prior.

Diskusi detail implementasi dan bacaan lebih lanjut ada di **Bab 09 Section 2.5** (peta keluarga generatif). Notebook ini berhenti di AE dan DAE - cukup untuk memberi pijakan sebelum membaca paper VAE, GAN, atau Diffusion.

## 8. Refleksi

1. Bandingkan kualitas t-SNE cluster AE standar vs DAE. Hipotesis mana yang cocok dengan hasil Anda: (a) DAE belajar fitur lebih robust karena harus mengabaikan noise, (b) DAE dan AE tidak berbeda secara substansial pada data sebersih CIFAR-10, atau (c) sesuatu yang lain?
2. Ambil satu kelas CIFAR-10 favorit. Lihat 5 gambar yang punya *reconstruction error* tertinggi di kelas itu. Apa pola visualnya? Apakah AE gagal pada gambar yang outlier dalam kelas, atau pada gambar yang tampak normal?
3. Jika Anda akan menambahkan *linear probe* (linear classifier di atas bottleneck frozen) untuk klasifikasi 10 kelas, akurasinya Anda perkirakan berapa persen? Uji. Apakah representasi unsupervised AE cukup kuat untuk klasifikasi, atau Anda butuh fine-tuning?